# FEM Pendulum: Contact

In this second tutorial on the FEM implementation of the pendulum, we extend the basic model by introducing a contact between the pendulum head and a wall.

1. **Basics**: We set up the basic model of a deformable pendulum without contact and external torque.
2. **Torque Application**: We extend the basic model by applying an external torque at the pivot point.
3. **Contact Handling** (this tutorial): We further extend the model by adding contact handling with a wall.

Following references were used to implement the model:
- [Nonlinear Elasticity - NGSolve 24](https://docu.ngsolve.org/ngs24/SaS/nonlinearelasticity.html)
- [Elastic Pendulum - NGS Tutorial 2024](https://docu.ngsolve.org/ngs24/tutorials/00_dynamics.html)
- [Contact Problems - NGS Docu Interactive Tutorial](https://docu.ngsolve.org/latest/i-tutorials/unit-6.2-contact/contact.html)
- [An Interactive Introduction to the Finite Element Method, Joachim Schöberl, TU Wien, ASC](https://jschoeberl.github.io/iFEM/intro.html)

```{figure} ../../figures/fem_pendulum.png
---
width: 60%
figclass: caption
alt: fem-pendulum-fig
name: fem-pendulum-fig
---
2D deformable pendulum with wall contact and torque application at the pivot point.
```

## Overview

This tutorial shows how a contact mechanism can be implemented for the pendulum. We will use the basic pendulum with torque application from the previous tutorial.

**Core ideas:**
1. We **add wall** as a second **geometry and material to our mesh**.
2. We define the **boundary conditions** and **adpat the bilinear form accordingly**.
2. We model the contact using a **contact boundary** between the pendulum head and the wall
3. We introduce a **gap function** which will track the gap and penetration of the contact bodies.
4. We use the gap function to **add an energy term** to the system equation which penalizes the penetration of the ball into the wall

**Assumptions:**
- Our goal is to achive energy conservation in the system.
- We neglect physical damping of the pendulum.

## Learning Goals

- Extend the previous 2D planar pendulum implementation by a energy-concerving contact model
- Understand how a contact can be modeled by adding an **energy term** to the system
- Understand the effect of **time step size** on the energy conservation in the dynamic simulation

## Prerequisites and Setup

This notebook assumes you understand the basic setup from [FEM Pendulum: Torque](02_fem_pendulum_torque.ipynb):

- 2D planar pendulum with out-of-plane thickness $h$
- hyperelastic (Neo-Hookean) material law
- hinge modeled by a mean-zero displacement constraint on the top edge `rotation`

Notes:
- All quantities are SI.
- We do **not** include damping or wall contact here (that is Tutorial 03).
- In the documentation build, notebooks are not executed (`nb_execution_mode = off`).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from netgen.occ import *
from ngsolve import *
from ngsolve.solvers import NewtonMinimization
from ngsolve.webgui import Draw